# 🏎️ F1 Strategy Predictor - FINAL PRODUCTION VERSION
## Real-Time Pit Stop & Compound Prediction

---

### 🎯 Obiettivo
**"Predict competitor pit stops in real-time to gain strategic advantage"**

### ✅ Caratteristiche Versione Finale
- **Unidirectional LSTM**: Non usa dati futuri (real-time compatible)
- **Temporal Split**: Train su gare passate, test su gare future
- **Features Ottimizzate**: Rimosse features inutili, aggiunti Circuit Type
- **Gradient Clipping**: Stabilità del training

### 📊 Target Metrics
- PIT Model: AUC-ROC ≥ 0.75 (honest, real-world)
- COMPOUND Model: Accuracy ≥ 0.70 (honest, real-world)

### Prerequisiti
- `f1_dataset_clean.pkl` (generato dal notebook DataAnalysis)

# 1. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os, sys
import json

pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [2]:
import joblib
import json
import os

os.makedirs('Model', exist_ok=True)
os.makedirs('Other', exist_ok=True)

In [3]:
# FastF1: libreria open-source per dati F1 (telemetria, tempi, meteo)
import importlib.util
if importlib.util.find_spec('fastf1') is None:
    !pip install fastf1 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.0/123.0 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.1 MB/s eta 0:00:00


In [4]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, roc_auc_score,
                             precision_recall_curve, roc_curve)

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout, BatchNormalization,
                                     Bidirectional, Masking, MultiHeadAttention,
                                     LayerNormalization, Add)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponibile: {tf.config.list_physical_devices('GPU')}")

# Riproducibilità
tf.random.set_seed(42)
np.random.seed(42)

TensorFlow version: 2.19.0
GPU disponibile: []


# 2. Import Dataset

In [6]:
# Carica il dataset già pulito (da notebook DataAnalysis)
# Le seguenti operazioni sono già state effettuate:
#   - Conversione Timedelta -> secondi (*Sec)
#   - Unificazione nomi team
#   - Rimozione compound non validi

df_f1 = pd.read_pickle('f1_dataset_clean.pkl')

print(f"Dataset caricato: {len(df_f1):,} righe")
print(f"\nTeam: {list(df_f1['Team'].unique())}")
print(f"\nCompound: {list(df_f1['Compound'].unique())}")

Dataset caricato: 86,757 righe

Team: ['Ferrari', 'Red Bull Racing', 'Mercedes', 'Haas F1 Team', 'Racing Bulls', 'Alpine', 'Williams', 'Kick Sauber', 'Aston Martin', 'McLaren']

Compound: ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']


# 3. Features + Target

---
## Targets

In [7]:
# Ordinamento cronologico: essenziale per sequenze temporali!
df_f1 = df_f1.sort_values(['Year', 'Round', 'Driver', 'LapNumber']).reset_index(drop=True)

**TARGET 1: PitIn3Laps** (Binary)
- 1 se pit entro i prossimi 3 giri, 0 altrimenti
- *Perché 3 giri?* Una finestra più ampia cattura l'imminenza del pit senza essere troppo generica. Un pit "imminente" a 1 giro è troppo restrittivo (poche sample positive).

In [8]:
# --- TARGET 1: PitIn3Laps ---

# Identifica i giri in cui il pilota pittarà entro i prossimi 3
df_f1['NextLapStint'] = df_f1.groupby(['Year', 'Round', 'Driver'])['Stint'].shift(-1)
# Lo shift -1 consente di spostare la colonna Stint verso l'alto = giro seguente
df_f1['IsPitLap'] = (df_f1['NextLapStint'] != df_f1['Stint']).astype(float).fillna(0)
# Fillna(0) sostituisce valori mancanti con 0. IsPtLap avrà valore 0/1.

# Aggregazione su finestra di 3 giri
df_f1['PitIn1'] = df_f1.groupby(['Year', 'Round', 'Driver'])['IsPitLap'].shift(-1).fillna(0)
# Lo shift -1 consente di spostare la colonna Stint verso l'alto = giro seguente
# Fillna(0) sostituisce valori NaN con 0 (ex. se considero ultimo giro)
df_f1['PitIn2'] = df_f1.groupby(['Year', 'Round', 'Driver'])['IsPitLap'].shift(-2).fillna(0)
df_f1['PitIn3'] = df_f1.groupby(['Year', 'Round', 'Driver'])['IsPitLap'].shift(-3).fillna(0)

df_f1['PitIn3Laps'] = ((df_f1['IsPitLap'] + df_f1['PitIn1'] + df_f1['PitIn2']
                        + df_f1['PitIn3']) > 0).astype(int)

In [9]:
# Pulizia colonne temporanee
df_f1 = df_f1.drop(columns=['PitIn1', 'PitIn2', 'PitIn3', 'NextLapStint'], errors='ignore')

print(f"Target PitIn3Laps: {df_f1['PitIn3Laps'].mean():.1%} positive")

Target PitIn3Laps: 20.7% positive


**TARGET 2: NextCompound** (Multiclass)
- Il compound che verrà montato al prossimo pit stop
- Classes: SOFT, MEDIUM, HARD, INTERMEDIATE, WET

In [10]:
# Per ogni giro, trova il compound del prossimo stint.
# Questo richiede di guardare avanti nella sequenza dei giri.

def get_next_compound(group):
    """
    Per ogni giro di un pilota, trova quale compound verrà montato al prossimo pit.
    Se il pilota non farà più pit, mantiene il compound attuale.
    """
    group = group.sort_values('LapNumber').copy()
    stints = group['Stint'].values
    compounds = group['StintCompound'].values
    result = compounds.copy()

    for i in range(len(group)):
        # Trova il prossimo stint diverso dall'attuale
        for j in range(i + 1, len(group)):
            if stints[j] != stints[i]:
                result[i] = compounds[j]
                break
    return result

In [11]:
# --- TARGET 2: NextCompound ---

# Prima creiamo il compound dello stint (primo compound usato nello stint)
df_f1['StintCompound'] = df_f1.groupby(['Year', 'Round', 'Driver', 'Stint'])['Compound'].transform('first')

# Applica la funzione per trovare il prossimo compound
next_compounds = []
for name, group in df_f1.groupby(['Year', 'Round', 'Driver']):
    next_compounds.extend(get_next_compound(group))

df_f1['NextCompound'] = next_compounds

In [12]:
# Pulizia colonne temporanee
df_f1 = df_f1.drop(columns=['StintCompound'], errors='ignore')

print(f"\nNextCompound distribution:")
print(df_f1['NextCompound'].value_counts())


NextCompound distribution:
NextCompound
HARD            45065
MEDIUM          24448
SOFT            13143
INTERMEDIATE     3971
WET               130
Name: count, dtype: int64


## Features

---
Per il Modello **PIT** (Quando pittare?)

| Categoria | Features | Tipo |
|-----------|----------|------|
| Temporali | LapNumber, Stint, StintProgress | RAW |
| Stato gomme | TyreLife, FreshTyre | RAW |
| Performance | LapTimeSec, Sector*TimeSec, Speed* | RAW |
| Condizioni | AirTemp, TrackTemp, Humidity, Pressure, WindSpeed | RAW |
| Safety | UnderSC, UnderVSC, UnderCaution | RAW |
| Gap | GapToLeader, Position | RAW |
| **Engineered** | TyreMargin, LapTimeTrend3, GapTrend, MyTyreVsField, MyStintVsField | **UNIQUE** |

In [13]:
FEATURES_PIT = [
    'LapNumber',
    'Stint',
    'Position',
    'TyreLife',
    'LapTimeSec',
    'SpeedST',
    'SpeedI1',
    'SpeedI2',
    'FreshTyre',
    'TrackTemp',
    'Pressure',
    'IsHard',
    'IsSoft',
    'TyreMargin',
    'LapTimeTrend3',
    'MyTyreVsField',
    'MyStintVsField',
    'IsStreetCircuit',
    'IsPowerCircuit',
    'IsHighDFCircuit',
]

print(f"FEATURES_PIT: {len(FEATURES_PIT)} features (optimized from 29)")

FEATURES_PIT: 20 features (optimized from 29)


---
Per il Modello **COMPOUND** (Quale gomma montare?)

**PURE RAW FEATURES** - I test hanno dimostrato che le raw features performano meglio delle engineered per questo task.

| Categoria | Features | Domanda |
|-----------|----------|----------|
| Compound attuale | IsSoft, IsMedium, IsHard, IsInter, IsWet | Cosa ho adesso? |
| Stato gara | LapNumber, Stint, Position, TyreLife | A che punto siamo? |
| Condizioni meteo | AirTemp, TrackTemp, Humidity, Pressure, WindSpeed, Rainfall | Serve gomma rain? |
| Safety | UnderSC, UnderVSC, UnderCaution | C'è safety car? |
| Stint | StintProgress, FreshTyre | Come sta la gomma? |

In [14]:
FEATURES_COMPOUND = [
    'LapNumber',
    'Stint',
    'AirTemp',
    'TyreLife',
    'LapTimeSec',
    'MyStintVsField',
    'MyTyreVsField',
    'SpeedST',
    'IsSoft',
    'IsMedium',
    'IsHard',
    'IsInter',
    'TeamEnc',
    'TrackTemp',
    'Rainfall',
    'Humidity',
    'AirTemp',
    'IsStreetCircuit',
    'IsPowerCircuit',
    'IsHighDFCircuit',
]

print(f"FEATURES_COMPOUND: {len(FEATURES_COMPOUND)} features (optimized from 20)")

FEATURES_COMPOUND: 20 features (optimized from 20)


---
# 4. Preparazione Dati per Training

### 4.1 Pulizia e Validazione

In [15]:
# Gestione valori infiniti (da divisioni nelle features calcolate)
df_f1 = df_f1.replace([np.inf, -np.inf], np.nan)

# Fill NaN nelle features con mediana
all_features = list(set(FEATURES_PIT + FEATURES_COMPOUND))
for f in all_features:
    if f in df_f1.columns:
        df_f1[f] = df_f1[f].fillna(df_f1[f].median())

In [16]:
# Filtraggio per training
df_clean = df_f1.dropna(subset=['PitIn3Laps', 'NextCompound'])  # Target validi

# Encoding target compound
label_encoder = LabelEncoder()
df_clean['NextCompoundEnc'] = label_encoder.fit_transform(df_clean['NextCompound'])
print(f"\nClassi: {list(label_encoder.classes_)}")


Classi: ['HARD', 'INTERMEDIATE', 'MEDIUM', 'SOFT', 'WET']


### 4.2 Creazione Sequenze per LSTM

*Perché sequenze di 10 giri?*
L'LSTM ha bisogno di contesto temporale per catturare i trend: 10 giri è un buon compromesso: abbastanza per vedere il degrado, non troppo per includere rumore. Stint più corti vengono paddati con valMedio (il Masking layer li ignorerà)

In [17]:
SEQUENCE_LENGTH = 10  # Giri di storia per la predizione

# Crea sequenze temporali per LSTM
def create_sequences(df, features, target_col, seq_len):
    """
    Crea sequenze temporali per LSTM.
    Input: (samples, timesteps, features) = (N, 10, 14)
    Per ogni giro, prende i 'seq_len' giri precedenti come input.
    """
    X = []  # Input = sequenza di giri
    y = []  # Output = target (PitIn3Laps o NextCompoundEnc)
    available = [f for f in features if f in df.columns] # Filtra features esistenti

    for (year, rnd, driver, stint), group in df.groupby(['Year', 'Round', 'Driver', 'Stint']):
        group = group.sort_values('LapNumber')
        if len(group) < 3:  # Skip stint troppo corti
            continue

        data = group[available].values.astype(np.float32)
        targets = group[target_col].values

        # Per ogni giro, prendi i 'seq_len' giri precedenti come input per creare la sequenza
        for i in range(1, len(group)):  # Inizia da 1 (serve almeno 1 giro di storia)
            start = max(0, i - seq_len)
            seq = data[start:i]

            # Padding se sequenza troppo corta (padding con zeri)
            if len(seq) < seq_len:
                pad = np.zeros((seq_len - len(seq), len(available)), dtype=np.float32)
                seq = np.vstack([pad, seq])

            X.append(seq)
            y.append(targets[i])

    return np.array(X), np.array(y), available

In [18]:
# Filtra features esistenti
FEATURES_PIT = [f for f in FEATURES_PIT if f in df_clean.columns]
FEATURES_COMPOUND = [f for f in FEATURES_COMPOUND if f in df_clean.columns]
print(f"Features PIT effettive: {len(FEATURES_PIT)}")
print(f"Features COMPOUND effettive: {len(FEATURES_COMPOUND)}")

# NOTA: Le sequenze verranno create DOPO lo split temporale
# per evitare data leakage tra train/val/test

Features PIT effettive: 20
Features COMPOUND effettive: 20


### 4.3 Split Train/Validation/Test (TEMPORAL)

In [19]:
# ============================================================================
# TEMPORAL SPLIT - For Real-Time Prediction
# ============================================================================
#
# WHY TEMPORAL? For real-time prediction during a race:
#   - You can only train on PAST races
#   - You predict on CURRENT/FUTURE races
#   - Random split would be cheating (seeing future patterns)
# ============================================================================

# Sort by time
df_clean = df_clean.sort_values(['Year', 'Round', 'LapNumber']).reset_index(drop=True)

# Create RaceID
df_clean['RaceID'] = df_clean['Year'].astype(str) + '_' + df_clean['Round'].astype(str).str.zfill(2)
unique_races = sorted(df_clean['RaceID'].unique())
n_races = len(unique_races)

# Temporal split: 70% train, 15% val, 15% test (chronologically)
train_cutoff = int(0.70 * n_races)
val_cutoff = int(0.85 * n_races)

train_races = set(unique_races[:train_cutoff])
val_races = set(unique_races[train_cutoff:val_cutoff])
test_races = set(unique_races[val_cutoff:])

print("="*60)
print("TEMPORAL SPLIT FOR REAL-TIME PREDICTION")
print("="*60)
print(f"\nTotal races: {n_races}")
print(f"\nSplit:")
print(f"  Train: {len(train_races)} races ({unique_races[0]} → {unique_races[train_cutoff-1]})")
print(f"  Val:   {len(val_races)} races ({unique_races[train_cutoff]} → {unique_races[val_cutoff-1]})")
print(f"  Test:  {len(test_races)} races ({unique_races[val_cutoff]} → {unique_races[-1]})")

# Split dataframes
df_train = df_clean[df_clean['RaceID'].isin(train_races)].copy()
df_val = df_clean[df_clean['RaceID'].isin(val_races)].copy()
df_test = df_clean[df_clean['RaceID'].isin(test_races)].copy()

print(f"\nLaps per split:")
print(f"  Train: {len(df_train):,} laps")
print(f"  Val:   {len(df_val):,} laps")
print(f"  Test:  {len(df_test):,} laps")

# Create sequences for each split
print("\nCreazione sequenze PIT...")
X_pit_train, y_pit_train, feat_pit = create_sequences(df_train, FEATURES_PIT, 'PitIn3Laps', SEQUENCE_LENGTH)
X_pit_val, y_pit_val, _ = create_sequences(df_val, FEATURES_PIT, 'PitIn3Laps', SEQUENCE_LENGTH)
X_pit_test, y_pit_test, _ = create_sequences(df_test, FEATURES_PIT, 'PitIn3Laps', SEQUENCE_LENGTH)

print(f"  Train: {X_pit_train.shape} | Positive: {y_pit_train.mean():.1%}")
print(f"  Val:   {X_pit_val.shape} | Positive: {y_pit_val.mean():.1%}")
print(f"  Test:  {X_pit_test.shape} | Positive: {y_pit_test.mean():.1%}")

print("\nCreazione sequenze COMPOUND...")
X_comp_train, y_comp_train, feat_comp = create_sequences(df_train, FEATURES_COMPOUND, 'NextCompoundEnc', SEQUENCE_LENGTH)
X_comp_val, y_comp_val, _ = create_sequences(df_val, FEATURES_COMPOUND, 'NextCompoundEnc', SEQUENCE_LENGTH)
X_comp_test, y_comp_test, _ = create_sequences(df_test, FEATURES_COMPOUND, 'NextCompoundEnc', SEQUENCE_LENGTH)

print(f"  Train: {X_comp_train.shape}")
print(f"  Val:   {X_comp_val.shape}")
print(f"  Test:  {X_comp_test.shape}")

print("\n✅ Temporal split completato!")

TEMPORAL SPLIT FOR REAL-TIME PREDICTION

Total races: 92

Split:
  Train: 64 races (2022_01 → 2024_20)
  Val:   14 races (2024_21 → 2025_10)
  Test:  14 races (2025_11 → 2025_24)

Laps per split:
  Train: 60,742 laps
  Val:   12,622 laps
  Test:  13,393 laps

Creazione sequenze PIT...
  Train: (57417, 10, 20) | Positive: 22.0%
  Val:   (11935, 10, 20) | Positive: 21.3%
  Test:  (12747, 10, 20) | Positive: 19.5%

Creazione sequenze COMPOUND...
  Train: (57417, 10, 20)
  Val:   (11935, 10, 20)
  Test:  (12747, 10, 20)

✅ Temporal split completato!


### 4.4 Normalizzazione e Class Weights

**Normalizzazione**: RobustScaler usa mediana invece di media
- *Perché?* Più robusto agli outliers (pit lap, safety car, outlap)

**Class Weights**: Bilancia le classi sbilanciate
- *Perché?* ~80% dei giri NON sono seguiti da pit -> il modello tenderebbe a predire sempre "no pit"

In [20]:
# Normalizzazione PIT
scaler_pit = RobustScaler()
n_features_pit = X_pit_train.shape[2] #   Shape: (81979, 10, 16)

X_pit_train = scaler_pit.fit_transform(X_pit_train.reshape(-1, n_features_pit)).reshape(X_pit_train.shape)
# = Shape: (81979, 10, 16) -> (819790, 16) -> norm -> (81979, 10, 16)
X_pit_val = scaler_pit.transform(X_pit_val.reshape(-1, n_features_pit)).reshape(X_pit_val.shape)
X_pit_test = scaler_pit.transform(X_pit_test.reshape(-1, n_features_pit)).reshape(X_pit_test.shape)


# Normalizzazione COMPOUND
scaler_comp = RobustScaler()
n_features_comp = X_comp_train.shape[2]   # Shape: (81979, 10, 15)

X_comp_train = scaler_comp.fit_transform(X_comp_train.reshape(-1, n_features_comp)).reshape(X_comp_train.shape)
# = Shape: (81979, 10, 15) -> (819790, 15) -> norm -> (81979, 10, 15)
X_comp_val = scaler_comp.transform(X_comp_val.reshape(-1, n_features_comp)).reshape(X_comp_val.shape)
X_comp_test = scaler_comp.transform(X_comp_test.reshape(-1, n_features_comp)).reshape(X_comp_test.shape)

In [21]:
# Class weights: weight = n_samples / (n_classes * n_samples_per_class)
# Classe rara -> weight alto

pit_counts = np.bincount(y_pit_train.astype(int))
pit_weight = {
    0: 1.0, # Classe di riferimento
    1: pit_counts[0] / pit_counts[1] # Classe rara
}
print(f"\nPit class weights:")
print(f"  Classe 0 (No Pit): {pit_weight[0]:.2f}")
print(f"  Classe 1 (Pit):    {pit_weight[1]:.2f}")

comp_counts = np.bincount(y_comp_train)
n_samples = len(y_comp_train)
comp_weight = { # non esiste una classe di riferimento
    0: n_samples / (5 * comp_counts[0]),
    1: n_samples / (5 * comp_counts[1]),
    2: n_samples / (5 * comp_counts[2]),
    3: n_samples / (5 * comp_counts[3]),
    4: n_samples / (5 * comp_counts[4]),
}
print(f"\nCompound class weights:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls:12s}: {comp_weight[i]:.2f} (n={comp_counts[i]:,})")


Pit class weights:
  Classe 0 (No Pit): 1.00
  Classe 1 (Pit):    3.54

Compound class weights:
  HARD        : 0.37 (n=30,658)
  INTERMEDIATE: 4.34 (n=2,647)
  MEDIUM      : 0.74 (n=15,580)
  SOFT        : 1.36 (n=8,434)
  WET         : 117.18 (n=98)


---
# 5. Modelli LSTM

### 5.1 Architettura Modello PIT

**Scelte architetturali:**

1. **Bidirectional LSTM**: Cattura pattern temporali in entrambe le direzioni
   - *Perché bidirezionale?* Il degrado può essere visibile sia guardando avanti che indietro nella sequenza

2. **Multi-Head Self-Attention**: Permette al modello di "pesare" quali giri sono più importanti
   - *Perché?* Un giro con degrado anomalo 5 giri fa potrebbe essere più predittivo del giro precedente

3. **Residual Connection**: Migliora il gradient flow
   - *Perché?* Permette ai gradienti di fluire direttamente, evitando vanishing gradient

4. **Masking**: Ignora i padding (zeri) nelle sequenze corte

In [22]:
def build_pit_model(seq_len, n_features):
    """
    PIT STOP PREDICTION MODEL - Final Production Version

    Architecture: Unidirectional LSTM + Attention
    - Unidirectional: Required for real-time (no future data)
    - Optimized regularization based on training analysis

    Input: (batch, seq_len, n_features)
    Output: (batch, 1) - probability of pit in next 3 laps
    """
    inputs = Input(shape=(seq_len, n_features), name='input')

    # Masking for padded sequences
    x = Masking(mask_value=0.0)(inputs)

    # LSTM Layer 1 - Unidirectional (real-time compatible)
    x = LSTM(128, return_sequences=True,
             dropout=0.3,              # Increased from 0.2
             recurrent_dropout=0.15)(x)  # Increased from 0.1
    x = LayerNormalization()(x)

    # Multi-Head Self-Attention
    attention = MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        dropout=0.1
    )(x, x)
    x = Add()([x, attention])  # Residual connection
    x = LayerNormalization()(x)

    # LSTM Layer 2
    x = LSTM(64, dropout=0.3, recurrent_dropout=0.15)(x)

    # Dense layers with regularization
    x = Dense(48, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)  # Increased from 0.3

    x = Dense(24, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.3)(x)  # Increased from 0.2

    # Output
    output = Dense(1, activation='sigmoid', name='pit')(x)

    model = Model(inputs=inputs, outputs=output)

    # Optimizer with gradient clipping for stability
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.001,
        clipnorm=1.0  # Gradient clipping
    )

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='prec'),
            tf.keras.metrics.Recall(name='rec')
        ]
    )

    return model

print("Building PIT model...")
model_pit = build_pit_model(SEQUENCE_LENGTH, len(feat_pit))
model_pit.summary()
print(f"\nFeatures used: {len(feat_pit)}")
print(f"Features: {feat_pit}")

Building PIT model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 10, 20)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 10, 20)    │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, 10, 20)    │          0 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 10)        │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 10, 128)   │     76,288 │ masking[0][0],    │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 128)   │        256 │ lstm[0][0]        │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 128)   │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logical_or          │ (None, 10)        │          0 │ any[0][0],        │
│ (LogicalOr)         │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 64)        │     49,408 │ layer_normalizat… │
│                     │                   │            │ logical_or[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 48)        │      3,120 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 48)        │        192 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 48)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 24)        │      1,176 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 24)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pit (Dense)         │ (None, 1)         │         25 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 196,769 (768.63 KB)

 Trainable params: 196,673 (768.25 KB)

 Non-trainable params: 96 (384.00 B)


Features used: 20
Features: ['LapNumber', 'Stint', 'Position', 'TyreLife', 'LapTimeSec', 'SpeedST', 'SpeedI1', 'SpeedI2', 'FreshTyre', 'TrackTemp', 'Pressure', 'IsHard', 'IsSoft', 'TyreMargin', 'LapTimeTrend3', 'MyTyreVsField', 'MyStintVsField', 'IsStreetCircuit', 'IsPowerCircuit', 'IsHighDFCircuit']


### 5.2 Training Modello PIT

In [ ]:
# Callbacks:
# - EarlyStopping: ferma se val_auc non migliora per 12 epoche
# - ReduceLROnPlateau: riduce learning rate se val_loss non migliora
callbacks_pit = [
    EarlyStopping(monitor='val_auc', patience=12, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print("Training PIT model...")
print("="*60)
history_pit = model_pit.fit(
    X_pit_train, y_pit_train,
    validation_data=(X_pit_val, y_pit_val),
    epochs=50,
    batch_size=128,
    callbacks=callbacks_pit,
    class_weight=pit_weight,
    verbose=1
)

Training PIT model...
Epoch 1/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 95s 179ms/step - accuracy: 0.5986 - auc: 0.6464 - loss: 1.1869 - prec: 0.3019 - rec: 0.6263 - val_accuracy: 0.6399 - val_auc: 0.7065 - val_loss: 0.6529 - val_prec: 0.3287 - val_rec: 0.6657 - learning_rate: 0.0010
Epoch 2/50
322/449 ━━━━━━━━━━━━━━━━━━━━ 17s 140ms/step - accuracy: 0.6403 - auc: 0.7208 - loss: 1.0302 - prec: 0.3478 - rec: 0.7284

### 5.3 Architettura Modello COMPOUND

**Architettura simile al PIT con output multiclass:**

1. **Stessa struttura BiLSTM + Attention** - cattura pattern temporali
2. **Output softmax** - per 5 classi mutualmente esclusive
3. **Pure raw features** - i test hanno mostrato che performano meglio degli engineered

In [ ]:
def build_compound_model(seq_len, n_features, n_classes):
    """
    COMPOUND PREDICTION MODEL - Final Production Version

    Architecture: Unidirectional LSTM + Attention
    - Lower learning rate to reduce training spikes
    - Optimized regularization

    Input: (batch, seq_len, n_features)
    Output: (batch, n_classes) - probability for each compound
    """
    inputs = Input(shape=(seq_len, n_features), name='input')

    x = Masking(mask_value=0.0)(inputs)

    # LSTM + Attention (same structure as PIT)
    x = LSTM(128, return_sequences=True,
             dropout=0.3, recurrent_dropout=0.15)(x)
    x = LayerNormalization()(x)

    attention = MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        dropout=0.1
    )(x, x)
    x = Add()([x, attention])
    x = LayerNormalization()(x)

    x = LSTM(64, dropout=0.3, recurrent_dropout=0.15)(x)

    x = Dense(48, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Dense(24, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.3)(x)

    # Output: softmax for multiclass
    output = Dense(n_classes, activation='softmax', name='compound')(x)

    model = Model(inputs=inputs, outputs=output)

    # Lower learning rate to prevent spikes seen in training
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.0005,  # Reduced from 0.001
        clipnorm=1.0
    )

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

print("Building COMPOUND model...")
n_classes = len(label_encoder.classes_)
model_comp = build_compound_model(SEQUENCE_LENGTH, len(feat_comp), n_classes)
model_comp.summary()
print(f"\nClasses: {list(label_encoder.classes_)}")
print(f"Features used: {len(feat_comp)}")

### 5.4 Training Modello COMPOUND

In [ ]:
callbacks_comp = [
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print("\nTraining COMPOUND model...")
print("="*60)
history_comp = model_comp.fit(
    X_comp_train, y_comp_train,
    validation_data=(X_comp_val, y_comp_val),
    epochs=50,
    batch_size=128,
    callbacks=callbacks_comp,
    class_weight=comp_weight,
    verbose=1
)

---
# 6. Valutazione

**Metriche PIT:**
- **AUC-ROC**: Metrica principale, misura la capacità di distinguere tra Pit e No Pit indipendentemente dalla soglia
- **F1 Score**: Bilancia Precision e Recall
- **Soglia ottimale**: Trovata massimizzando F1 sulla curva precision-recall

**Metriche COMPOUND:**
- **Accuracy**: Proporzione di predizioni corrette
- **F1 (weighted)**: Media pesata degli F1 per classe
- **Per-class accuracy**: Per identificare classi problematiche

In [ ]:
# ------ MODELLO PIT ------
print("\n" + "-"*50)
print("MODELLO PIT (PitIn3Laps)")
print("-"*50)

# Predizioni
y_pit_pred_prob = model_pit.predict(X_pit_test, verbose=0).flatten()

# Trova soglia ottimale (massimizza F1)
prec, rec, thresh = precision_recall_curve(y_pit_test, y_pit_pred_prob)
f1_arr = 2 * prec * rec / (prec + rec + 1e-8)
best_idx = np.argmax(f1_arr)
opt_thresh = thresh[best_idx] if best_idx < len(thresh) else 0.5

y_pit_pred = (y_pit_pred_prob > opt_thresh).astype(int)

# Metriche
pit_auc = roc_auc_score(y_pit_test, y_pit_pred_prob)
pit_f1 = f1_score(y_pit_test, y_pit_pred)
pit_acc = accuracy_score(y_pit_test, y_pit_pred)

print(f"AUC-ROC:   {pit_auc:.3f}")
print(f"F1 Score:  {pit_f1:.3f}")
print(f"Accuracy:  {pit_acc:.3f}")
print(f"Precision: {prec[best_idx]:.3f}")
print(f"Recall:    {rec[best_idx]:.3f}")
print(f"Soglia:    {opt_thresh:.3f}")

cm_pit = confusion_matrix(y_pit_test, y_pit_pred)
print(f"\nConfusion Matrix:")
print(f"           Pred:0  Pred:1")
print(f"True:0     {cm_pit[0,0]:6d}  {cm_pit[0,1]:6d}")
print(f"True:1     {cm_pit[1,0]:6d}  {cm_pit[1,1]:6d}")

In [ ]:
# ------ MODELLO COMPOUND ------
print("\n" + "-"*50)
print("MODELLO COMPOUND (NextCompound)")
print("-"*50)

y_comp_pred_prob = model_comp.predict(X_comp_test, verbose=0)
y_comp_pred = y_comp_pred_prob.argmax(axis=1)

comp_acc = accuracy_score(y_comp_test, y_comp_pred)
comp_f1 = f1_score(y_comp_test, y_comp_pred, average='weighted')

print(f"Accuracy:  {comp_acc:.3f}")
print(f"F1 (weighted): {comp_f1:.3f}")

# Per-class accuracy
print("\nPer-class accuracy:")
for i, cls in enumerate(label_encoder.classes_):
    mask = y_comp_test == i
    if mask.sum() > 0:
        cls_acc = (y_comp_pred[mask] == i).mean()
        cls_pred_count = (y_comp_pred == i).sum()
        print(f"  {cls:12s}: {cls_acc:.3f} (n={mask.sum():,}, pred={cls_pred_count:,})")

print(f"\nConfusion Matrix:")
cm_comp = confusion_matrix(y_comp_test, y_comp_pred)
print(f"Classes: {list(label_encoder.classes_)}")
print(cm_comp)

### 6.1 Feature Importance

*Come funziona?* Per ogni feature, si permutano casualmente i suoi valori e si misura quanto peggiora la performance (estensione del concetto di permutation importance ai modelli neurali).

In [ ]:
def permutation_importance(model, X, y, features, n_repeats=3, metric='auc'):
    """Calcola l'importanza delle features tramite permutation."""
    # Baseline performance
    y_pred = model.predict(X, verbose=0)
    if metric == 'auc':
        baseline = roc_auc_score(y, y_pred.flatten())
    else:
        baseline = accuracy_score(y, y_pred.argmax(axis=1))

    importances = {}
    for i, feat in enumerate(features):
        scores = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            # Permuta la feature i su tutti i timestep
            perm_idx = np.random.permutation(len(X_perm))
            X_perm[:, :, i] = X_perm[perm_idx, :, i]

            y_pred_perm = model.predict(X_perm, verbose=0)
            if metric == 'auc':
                score = roc_auc_score(y, y_pred_perm.flatten())
            else:
                score = accuracy_score(y, y_pred_perm.argmax(axis=1))
            scores.append(score)

        importances[feat] = baseline - np.mean(scores)

    return dict(sorted(importances.items(), key=lambda x: x[1], reverse=True))

In [ ]:
print("Calcolo feature importance PIT...")
pit_importance = permutation_importance(model_pit, X_pit_test, y_pit_test, feat_pit, metric='auc')

print("\nTop 10 features PIT:")
for feat, imp in list(pit_importance.items())[:10]:
    print(f"  {feat:20s}: {imp:.4f}")

In [ ]:
print("\nTop 10 features PIT:")
for feat, imp in list(pit_importance.items())[:50]:
    print(f"  {feat:20s}: {imp:.4f}")

In [ ]:
print("Calcolo feature importance COMPOUND...")
comp_importance = permutation_importance(model_comp, X_comp_test, y_comp_test, feat_comp, metric='accuracy')

print("\nTop 10 features COMPOUND:")
for feat, imp in list(comp_importance.items())[:10]:
    print(f"  {feat:20s}: {imp:.4f}")

In [ ]:
print("\nTop 10 features COMPOUND:")
for feat, imp in list(comp_importance.items())[:50]:
    print(f"  {feat:20s}: {imp:.4f}")

### 6.2 Visualizzazione Training History

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PIT - Loss
axes[0, 0].plot(history_pit.history['loss'], label='Train', linewidth=2)
axes[0, 0].plot(history_pit.history['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_title('PIT Model - Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# PIT - AUC
axes[0, 1].plot(history_pit.history['auc'], label='Train', linewidth=2)
axes[0, 1].plot(history_pit.history['val_auc'], label='Validation', linewidth=2)
axes[0, 1].axhline(y=pit_auc, color='r', linestyle='--', label=f'Test AUC: {pit_auc:.3f}')
axes[0, 1].set_title('PIT Model - AUC', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# COMPOUND - Loss
axes[1, 0].plot(history_comp.history['loss'], label='Train', linewidth=2)
axes[1, 0].plot(history_comp.history['val_loss'], label='Validation', linewidth=2)
axes[1, 0].set_title('COMPOUND Model - Loss', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# COMPOUND - Accuracy
axes[1, 1].plot(history_comp.history['accuracy'], label='Train', linewidth=2)
axes[1, 1].plot(history_comp.history['val_accuracy'], label='Validation', linewidth=2)
axes[1, 1].axhline(y=comp_acc, color='r', linestyle='--', label=f'Test Acc: {comp_acc:.3f}')
axes[1, 1].set_title('COMPOUND Model - Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Other/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Visualizzazione Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PIT feature importance
pit_top10 = dict(list(pit_importance.items())[:10])
colors_pit = ['#2ecc71' if 'Vs' in f or 'Trend' in f or 'Margin' in f else '#3498db' for f in pit_top10.keys()]
axes[0].barh(list(pit_top10.keys())[::-1], list(pit_top10.values())[::-1], color=colors_pit[::-1])
axes[0].set_xlabel('Importance (AUC drop)')
axes[0].set_title('PIT Model - Top 10 Features', fontweight='bold')
axes[0].axvline(x=0, color='black', linewidth=0.5)

# COMPOUND feature importance
comp_top10 = dict(list(comp_importance.items())[:10])
axes[1].barh(list(comp_top10.keys())[::-1], list(comp_top10.values())[::-1], color='#e74c3c')
axes[1].set_xlabel('Importance (Accuracy drop)')
axes[1].set_title('COMPOUND Model - Top 10 Features', fontweight='bold')
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('Other/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🟢 Green = Engineered features (unique information)")
print("🔵 Blue = Raw features")

---
# 7. Save

In [ ]:
# Salva modelli Keras
model_pit.save('Model/f1_pit_model.keras')
model_comp.save('Model/f1_compound_model.keras')
print("✓ Modelli salvati")

In [ ]:
# Salva scaler e encoder
joblib.dump(scaler_pit, 'Model/f1_pit_scaler.pkl')
joblib.dump(scaler_comp, 'Model/f1_comp_scaler.pkl')
joblib.dump(label_encoder, 'Model/label_encoder.pkl')
print("✓ Scaler e encoder salvati")

In [ ]:
# Salva configurazione
config = {
    'sequence_length': SEQUENCE_LENGTH,
    'features_pit': feat_pit,
    'features_pit_engineered': ['TyreMargin', 'LapTimeTrend3', 'GapTrend', 'MyTyreVsField', 'MyStintVsField'],
    'features_compound': feat_comp,
    'pit_threshold': float(opt_thresh),
    'compound_classes': list(label_encoder.classes_),
    'philosophy': 'minimal_engineering',
    'metrics': {
        'pit_auc': float(pit_auc),
        'pit_f1': float(pit_f1),
        'pit_accuracy': float(pit_acc),
        'compound_accuracy': float(comp_acc),
        'compound_f1': float(comp_f1)
    }
}

with open('Model/modelConfig.json', 'w') as f:
    json.dump(config, f, indent=2)
print("✓ Configurazione salvata")

In [ ]:
# Salva il dataset completo con tutte le features
df_f1.to_pickle('Other/f1_dataset_features.pkl')
print("✓ Dataset con features salvato")

print("\n" + "="*60)
print("TRAINING COMPLETATO!")
print("="*60)
print(f"\nPIT Model:      AUC={pit_auc:.3f}, F1={pit_f1:.3f}")
print(f"COMPOUND Model: Accuracy={comp_acc:.3f}, F1={comp_f1:.3f}")